# RecruTap - Feature Engineering

## Objective

This notebook transforms the cleaned job posting dataset into machine-learning-ready features.

The objectives are:

- Load the cleaned dataset
- Combine important text fields
- Clean and normalize text
- Create a single NLP feature
- Save the processed dataset for model training

In [1]:
import pandas as pd
import numpy as np

import re
import string

from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

In [2]:
df = pd.read_csv("../data/processed/clean_job_postings.csv")

df.head()

,job_id,title,location,company_profile,description,requirements,benefits,telecommuting,has_company_logo,has_questions,employment_type,required_experience,required_education,industry,function,fraudulent
0,1,Marketing Intern,"US, NY, New York","We're Food52, and we've created a groundbreaki...","Food52, a fast-growing, James Beard Award-winn...",Experience with content management systems a m...,NaN,0,1,0,Other,Internship,Unknown,Unknown,Marketing,0
1,2,Customer Service - Cloud Video Production,"NZ, , Auckland","90 Seconds, the worlds Cloud Video Production ...",Organised - Focused - Vibrant - Awesome!Do you...,What we expect from you:Your key responsibilit...,What you will get from usThrough being part of...,0,1,0,Full-time,Not Applicable,Unknown,Marketing and Advertising,Customer Service,0
2,3,Commissioning Machinery Assistant (CMA),"US, IA, Wever",Valor Services provides Workforce Solutions th...,"Our client, located in Houston, is actively se...",Implement pre-commissioning and commissioning ...,NaN,0,1,0,Unknown,Unknown,Unknown,Unknown,Unknown,0
3,4,Account Executive - Washington DC,"US, DC, Washington",Our passion for improving quality of life thro...,THE COMPANY: ESRI – Environmental Systems Rese...,"EDUCATION: Bachelor’s or Master’s in GIS, busi...",Our culture is anything but corporate—we have ...,0,1,0,Full-time,Mid-Senior level,Bachelor's Degree,Computer Software,Sales,0
4,5,Bill Review Manager,"US, FL, Fort Worth",SpotSource Solutions LLC is a Global Human Cap...,JOB TITLE: Itemization Review ManagerLOCATION:...,QUALIFICATIONS:RN license in the State of Texa...,Full Benefits Offered,0,1,1,Full-time,Mid-Senior level,Bachelor's Degree,Hospital & Health Care,Health Care Provider,0


## Why Feature Engineering?

Machine learning algorithms cannot directly interpret raw textual information.

Feature engineering converts unstructured job posting text into structured representations that preserve meaningful information while removing unnecessary noise.

This stage serves as the bridge between data preprocessing and model training.

In [3]:
df.shape

(17879, 16)

## Selecting Relevant Features

For fraud detection, textual information contains the strongest predictive signals.

### Combining Textual Features

Each job posting contains information distributed across multiple columns such as:

- Job Title
- Company Profile
- Description
- Requirements
- Benefits

Since these collectively describe the job posting, they are merged into a single text feature that captures the overall context.

In [4]:
text_columns = [
    "title",
    "company_profile",
    "description",
    "requirements",
    "benefits"
]

In [5]:
df["text"] = df[text_columns].fillna("").agg(" ".join, axis=1)

In [6]:
df["text"].head()

0    Marketing Intern We're Food52, and we've creat...
1    Customer Service - Cloud Video Production 90 S...
2    Commissioning Machinery Assistant (CMA) Valor ...
3    Account Executive - Washington DC Our passion ...
4    Bill Review Manager SpotSource Solutions LLC i...
Name: text, dtype: object

## Text Normalization

Raw text often contains unnecessary elements that provide little predictive value.

The preprocessing pipeline includes:

- Converting text to lowercase
- Removing URLs
- Removing punctuation
- Removing numerical values
- Removing English stop words

These steps improve consistency while reducing noise for downstream NLP models.

In [7]:
def clean_text(text):

    text = text.lower()

    text = re.sub(r"http\S+", "", text)

    text = re.sub(r"\d+", "", text)

    text = text.translate(
        str.maketrans("", "", string.punctuation)
    )

    words = text.split()

    words = [
        word
        for word in words
        if word not in ENGLISH_STOP_WORDS
    ]

    return " ".join(words)

## Processed Text Preview

The cleaned text is compared with the original combined text to verify that important information has been preserved while irrelevant tokens have been removed.

In [8]:
df["clean_text"] = df["text"].apply(clean_text)

In [9]:
df[["text", "clean_text"]].head()

,text,clean_text
0,"Marketing Intern We're Food52, and we've creat...",marketing intern food weve created groundbreak...
1,Customer Service - Cloud Video Production 90 S...,customer service cloud video production second...
2,Commissioning Machinery Assistant (CMA) Valor ...,commissioning machinery assistant cma valor se...
3,Account Executive - Washington DC Our passion ...,account executive washington dc passion improv...
4,Bill Review Manager SpotSource Solutions LLC i...,review manager spotsource solutions llc global...


In [10]:
df = df[df["clean_text"].str.strip() != ""]

In [ ]:
model_df = df[["clean_text", "fraudulent"]]

In [12]:
model_df.head()

,clean_text,fraudulent
0,marketing intern food weve created groundbreak...,0
1,customer service cloud video production second...,0
2,commissioning machinery assistant cma valor se...,0
3,account executive washington dc passion improv...,0
4,review manager spotsource solutions llc global...,0


## Preparing the Final Dataset

The final machine learning dataset contains:

- clean_text → input feature
- fraudulent → target variable

This dataset is exported for TF-IDF vectorization and model training.

In [13]:
model_df.to_csv("../data/processed/model_dataset.csv",index=False)

# Conclusion

The feature engineering pipeline successfully transformed multiple textual attributes into a single normalized feature suitable for Natural Language Processing.

The processed dataset is now prepared for:

- TF-IDF Vectorization
- Model Training
- Fraud Prediction
- Backend Integration